# Data preparation for the Text2Hazard experiment

This notebook prepares the cleaned document corpus and selects the documents for manual evaluation. It uses the reviewed metadata and the findings from Notebook 01.

## 1. Import the libraries

In [1]:
from pathlib import Path
import re
import zipfile

import pandas as pd

# Find the project folder
PROJECT_FOLDER = Path.cwd()

candidate_folders = [
    PROJECT_FOLDER,
    PROJECT_FOLDER.parent,
    PROJECT_FOLDER.parent.parent,
]

for folder in candidate_folders:
    if (folder / "data/text_2_data.zip").exists():
        PROJECT_FOLDER = folder
        break

ZIP_PATH = PROJECT_FOLDER / "data/text_2_data.zip"
METADATA_PATH = PROJECT_FOLDER / "data/text2data_document_metadata.xlsx"
OUTPUT_PATH = PROJECT_FOLDER / "data/prepared_documents.csv"

print(f"ZIP: {ZIP_PATH}")
print(f"Metadata: {METADATA_PATH}")
print(f"Prepared CSV: {OUTPUT_PATH}")

ZIP: C:\Users\onssr\Desktop\AIHWS\text2hazard_project\data\text_2_data.zip
Metadata: C:\Users\onssr\Desktop\AIHWS\text2hazard_project\data\text2data_document_metadata.xlsx
Prepared CSV: C:\Users\onssr\Desktop\AIHWS\text2hazard_project\data\prepared_documents.csv


## 2. Helper functions

I use the same word-count and filename-matching functions as in Notebook 01.

In [2]:
def count_words(text):
    # Count words made of letters
    pattern = r"\b[^\W\d_]+(?:['’\-][^\W\d_]+)*\b"
    return len(re.findall(pattern, text, flags=re.UNICODE))


def cleaned_document_name(filename):
    return re.sub(r"_cleaned\.md$", "", Path(filename).name, flags=re.I)

## 3. Read the cleaned text and reviewed metadata

The model input is the cleaned Markdown text. The reviewed Excel table provides the source type and language information.

In [ ]:
# Load only the reviewed metadata fields used in this notebook
metadata = pd.read_excel(METADATA_PATH)

metadata_columns = [
    "document_name",
    "document_type",
    "cleaned_language",
    "original_source_language",
]

missing_columns = set(metadata_columns) - set(metadata.columns)
if missing_columns:
    raise ValueError(f"Missing metadata columns: {sorted(missing_columns)}")

metadata = metadata[metadata_columns].copy()

# Read the cleaned texts directly from the ZIP archive
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    files = [item for item in archive.infolist() if not item.is_dir()]
    pdf_files = [item for item in files if item.filename.lower().endswith(".pdf")]
    markdown_files = [item for item in files if item.filename.lower().endswith(".md")]

    rows = []
    for item in markdown_files:
        text = archive.read(item).decode("utf-8-sig", errors="replace")
        rows.append({
            "document_name": cleaned_document_name(item.filename),
            "word_count": count_words(text),
        })

cleaned_documents = pd.DataFrame(rows).merge(
    metadata,
    on="document_name",
    how="left",
    validate="one_to_one",
)

if cleaned_documents["document_type"].isna().any():
    raise ValueError("At least one cleaned document did not match the reviewed metadata.")

print(f"Original PDFs: {len(pdf_files)}")
print(f"Cleaned Markdown documents: {len(cleaned_documents)}")
print(f"Reviewed metadata rows: {len(metadata)}")
display(cleaned_documents.head(3))

Original PDFs: 77
Cleaned Markdown documents: 76
Reviewed metadata rows: 76


,document_name,word_count,document_type,cleaned_language,original_source_language
0,ABC 2024 - Traffic jams and flight delays due ...,706,News / media,English,English
1,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,10477,Institutional / technical report,English,Spanish
2,AFP 2022 - The_Vibes_Valencia Airport in Madri...,327,News / media,English,English


## 4. Exclude the PDF without cleaned Markdown

The Papatolios document has an original PDF but no cleaned Markdown file. It is excluded from the experimental corpus because both LLM workflows require cleaned Markdown as input. The original PDF remains unchanged in the ZIP archive.

In [4]:
# Compare PDF and Markdown filename stems
pdf_names = {
    Path(item.filename).stem.lower(): Path(item.filename).name
    for item in pdf_files
}

markdown_stems = {
    re.sub(r"_cleaned$", "", Path(item.filename).stem.lower())
    for item in markdown_files
}

missing_markdown_files = [
    original_name
    for stem, original_name in pdf_names.items()
    if stem not in markdown_stems
]

assert len(missing_markdown_files) == 1
assert missing_markdown_files[0].startswith("Papatolios 2021")

display(pd.DataFrame({"PDF without cleaned Markdown": missing_markdown_files}))

,PDF without cleaned Markdown
0,Papatolios 2021 - Belgium-Germany_ estimated c...


In [5]:
# Start the preparation table with the 76 cleaned documents
prepared = cleaned_documents.rename(
    columns={"document_type": "source_type"}
)[["document_name", "source_type", "word_count"]].copy()

prepared["markdown_available"] = True

# Keep the missing PDF visible in the table, but mark it as excluded
missing_row = pd.DataFrame([{
    "document_name": Path(missing_markdown_files[0]).stem,
    "source_type": pd.NA,
    "word_count": pd.NA,
    "markdown_available": False,
}])

prepared = pd.concat([prepared, missing_row], ignore_index=True)
prepared["word_count"] = prepared["word_count"].astype("Int64")
prepared["duplicate_status"] = "not a confirmed duplicate"
prepared["experimental_corpus_inclusion"] = prepared["markdown_available"]
prepared["length_outlier"] = False
prepared["evaluation_eligibility"] = False
prepared["evaluation_selection"] = False
prepared["evaluation_length_position"] = ""
prepared["exclusion_reason"] = ""

missing_mask = ~prepared["markdown_available"]
prepared.loc[missing_mask, "duplicate_status"] = "not assessed (no Markdown)"
prepared.loc[missing_mask, "exclusion_reason"] = "No cleaned Markdown input"

print(f"Documents with cleaned Markdown: {prepared['markdown_available'].sum()}")
print(f"Documents excluded because Markdown is missing: {missing_mask.sum()}")

Documents with cleaned Markdown: 76
Documents excluded because Markdown is missing: 1


## 5. Handle the confirmed duplicate

The two Storm Desmond files were manually checked against the original documents and are genuine duplicates. I retain the Treanor version because its filename preserves the author name and its cleaned text contains the same article body with slightly less page and navigation text. Both original files remain unchanged in the ZIP archive.

In [6]:
# Find the two manually confirmed copies
storm_desmond_mask = prepared["document_name"].str.contains(
    "Storm Desmond damage across Cumbria estimated",
    regex=False,
)

storm_desmond_pair = prepared.loc[
    storm_desmond_mask,
    ["document_name", "source_type", "word_count"],
].copy()

display(storm_desmond_pair)

,document_name,source_type,word_count
64,The Guardian 2015 - Storm Desmond damage acros...,News / media,917
68,Treanor 2015 - Storm Desmond damage across Cum...,News / media,904


In [7]:
treanor_document = prepared.loc[
    prepared["document_name"].str.startswith("Treanor 2015 - Storm Desmond damage across"),
    "document_name",
].item()

guardian_duplicate = prepared.loc[
    prepared["document_name"].str.startswith("The Guardian 2015 - Storm Desmond damage across"),
    "document_name",
].item()

# Record which copy is retained and which is excluded
prepared.loc[
    prepared["document_name"] == treanor_document,
    "duplicate_status",
] = "confirmed duplicate - retained"

excluded_duplicate_mask = prepared["document_name"] == guardian_duplicate
prepared.loc[excluded_duplicate_mask, "duplicate_status"] = "confirmed duplicate - excluded"
prepared.loc[excluded_duplicate_mask, "experimental_corpus_inclusion"] = False
prepared.loc[excluded_duplicate_mask, "exclusion_reason"] = (
    "Confirmed duplicate; Treanor 2015 retained"
)

print(f"Retained: {treanor_document}")
print(f"Excluded duplicate: {guardian_duplicate}")

Retained: Treanor 2015 - Storm Desmond damage across Cumbria estimated at £500m _ Storm Desmond _The Guardian
Excluded duplicate: The Guardian 2015 - Storm Desmond damage across Cumbria estimated at £500m _ Storm Desmond


## 6. Apply the evaluation outlier rule

Notebook 01 identified eight potential document-length outliers using the 1.5×IQR rule. I reuse that result here and do not calculate a new threshold after excluding the missing document or duplicate.

The eight documents remain part of the experimental corpus. Following the feedback received after the project presentation, they are excluded only from the candidate pool for the manually annotated quantitative evaluation sample. No chunking decision is made here.

In [8]:
# Fixed result from Notebook 01; the IQR is not recalculated here
OUTLIER_DOCUMENTS = {
    "IPCC 2018 - Working Group 2 Cross-chapter case studies",
    "Fekete 2025 - Cascading impact chains and recovery challenges of the 2024 Valencia catastrophic floods",
    "McGregor 2007 - The social impacts of heat waves",
    "Mohr 2022 - A multi-disciplinary analysis of the exceptional flood event of July 2021 in central Europe - Part 1 Event desciption and analysis",
    "Eurelectric 2006 - Impacts of Severe Storms on Electric Grids",
    "Deidda 2025 - Is Europe's transport infrastructure ready to face climate change",
    "Hladny 2004 - August_2002_catastrophic_flood_in_the_Czech_Republic",
    "Tapsell 2010 - Social vulnerability to natural hazards",
}

prepared["length_outlier"] = prepared["document_name"].isin(OUTLIER_DOCUMENTS)

outlier_table = prepared.loc[
    prepared["length_outlier"],
    ["document_name", "source_type", "word_count", "experimental_corpus_inclusion"],
].sort_values("word_count")

assert len(outlier_table) == 8
assert outlier_table["experimental_corpus_inclusion"].all()

display(outlier_table)

,document_name,source_type,word_count,experimental_corpus_inclusion
28,IPCC 2018 - Working Group 2 Cross-chapter case...,Institutional / technical report,10907,True
20,Fekete 2025 - Cascading impact chains and reco...,Scientific paper,11444,True
45,McGregor 2007 - The social impacts of heat waves,Scientific paper,11639,True
48,Mohr 2022 - A multi-disciplinary analysis of t...,Scientific paper,11690,True
16,Eurelectric 2006 - Impacts of Severe Storms on...,Institutional / technical report,13074,True
13,Deidda 2025 - Is Europe's transport infrastruc...,Scientific paper,14629,True
27,Hladny 2004 - August_2002_catastrophic_flood_i...,Institutional / technical report,15880,True
62,Tapsell 2010 - Social vulnerability to natural...,Scientific paper,25888,True


In [9]:
# Outliers stay in the corpus but cannot enter the evaluation sample
prepared["evaluation_eligibility"] = (
    prepared["experimental_corpus_inclusion"]
    & ~prepared["length_outlier"]
)

outlier_reason_mask = prepared["length_outlier"]
prepared.loc[outlier_reason_mask, "exclusion_reason"] = (
    "IQR length outlier; excluded from evaluation sampling only"
)

status_summary = pd.DataFrame({
    "status": [
        "Original PDFs",
        "Cleaned Markdown documents",
        "Experimental corpus",
        "Eligible for evaluation sampling",
    ],
    "documents": [
        len(pdf_files),
        int(prepared["markdown_available"].sum()),
        int(prepared["experimental_corpus_inclusion"].sum()),
        int(prepared["evaluation_eligibility"].sum()),
    ],
})

display(status_summary)

,status,documents
0,Original PDFs,77
1,Cleaned Markdown documents,76
2,Experimental corpus,75
3,Eligible for evaluation sampling,67


## 7. Select the evaluation sample

The eligible documents are grouped by reviewed source type and ordered by word count. For each source type, I select the shortest document, the middle-ranked document, and the longest document.

For groups with an even number of documents, the existing convention is used: the upper of the two central positions is selected. The final nine documents are deliberately selected to cover source type and document length. They are not statistically representative of the complete corpus.

In [10]:
eligible_documents = prepared.loc[
    prepared["evaluation_eligibility"]
].copy()

eligible_by_type = (
    eligible_documents.groupby("source_type")
    .size()
    .rename("eligible_documents")
    .reset_index()
)

display(eligible_by_type)

,source_type,eligible_documents
0,Institutional / technical report,8
1,News / media,40
2,Scientific paper,19


In [11]:
selected_rows = []

# Select the three length positions separately within each source type
for source_type, group in eligible_documents.groupby("source_type"):
    group = group.sort_values(["word_count", "document_name"]).reset_index(drop=True)

    # Integer division selects the upper middle position for even groups
    selected_indices = [0, len(group) // 2, len(group) - 1]
    positions = ["short", "middle", "long"]

    for position, index in zip(positions, selected_indices):
        row = group.iloc[index].copy()
        row["evaluation_length_position"] = position
        selected_rows.append(row)

evaluation_sample = pd.DataFrame(selected_rows)

prepared.loc[
    prepared["document_name"].isin(evaluation_sample["document_name"]),
    "evaluation_selection",
] = True

position_by_document = evaluation_sample.set_index("document_name")[
    "evaluation_length_position"
]
prepared["evaluation_length_position"] = (
    prepared["document_name"].map(position_by_document).fillna("")
)

position_order = {"short": 0, "middle": 1, "long": 2}
source_order = {
    "Institutional / technical report": 0,
    "News / media": 1,
    "Scientific paper": 2,
}

evaluation_sample = prepared.loc[
    prepared["evaluation_selection"],
    ["document_name", "source_type", "word_count", "evaluation_length_position"],
].copy()

evaluation_sample["source_order"] = evaluation_sample["source_type"].map(source_order)
evaluation_sample["position_order"] = evaluation_sample["evaluation_length_position"].map(position_order)
evaluation_sample = evaluation_sample.sort_values(
    ["source_order", "position_order"]
).drop(columns=["source_order", "position_order"])

display(evaluation_sample.reset_index(drop=True))

,document_name,source_type,word_count,evaluation_length_position
0,PWC 2015 - Updated estimates on cost of Storm ...,Institutional / technical report,484,short
1,Krausmann 2014 - STREST report on lessons lear...,Institutional / technical report,1469,middle
2,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,Institutional / technical report,10477,long
3,Lavenir 2021 - À la recherche d'une jeune fill...,News / media,157,short
4,ABC 2024 - Traffic jams and flight delays due ...,News / media,706,middle
5,"EFE 2024 - The DANA storm, live_ The death tol...",News / media,3635,long
6,Plag 2014 - Foreword extreme geohazards—a grow...,Scientific paper,1457,short
7,Kadir 2014 - The Impact of Natural Disasters o...,Scientific paper,4935,middle
8,"Cavaleri 2019 - The October 29, 2018 storm in ...",Scientific paper,9218,long


## 8. Reference annotations

After selecting the nine evaluation documents, I created a structured reference annotation table for their hazards, locations, times, and impacts. Documents without a qualifying event are also included. This CSV will be used later to evaluate the two LLM extraction workflows.

In [12]:
ANNOTATION_PATH = PROJECT_FOLDER / "data/evaluation/manual_annotations.csv"

# Load the finalized manual reference annotations
annotations = pd.read_csv(ANNOTATION_PATH, keep_default_na=False)
display(annotations.head())

,document_id,event_id,fact_type,category,source_text,normalized_name,country_code,normalized_start,normalized_end,description,evidence_quote,document_notes
0,PWC 2015 - Updated estimates on cost of Storm ...,event_01,hazard,storm,Storm Desmond,,,,,,Our current estimate of the damage caused by S...,"[""The 2009 floods are treated as a historical ..."
1,PWC 2015 - Updated estimates on cost of Storm ...,event_01,hazard,flood,flood affected areas,,,,,,Insurers of affected policyholders have mobili...,"[""The 2009 floods are treated as a historical ..."
2,PWC 2015 - Updated estimates on cost of Storm ...,event_01,impact,economic,,,,,,damage estimated at £400m - £500m,Our current estimate of the damage caused by S...,"[""The 2009 floods are treated as a historical ..."
3,PWC 2015 - Updated estimates on cost of Storm ...,event_01,impact,economic,,,,,,insurance industry paying out between £250m an...,Our current estimate of the damage caused by S...,"[""The 2009 floods are treated as a historical ..."
4,Krausmann 2014 - STREST report on lessons lear...,event_01,hazard,other,Great East Japan earthquake,,,,,,The Great East Japan earthquake and the tsunam...,"[""The Great East Japan earthquake and the tsun..."


In [13]:
fact_types = ["hazard", "location", "time", "impact"]

annotation_summary = (
    annotations.loc[annotations["fact_type"].isin(fact_types)]
    .groupby(["document_id", "fact_type"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=fact_types, fill_value=0)
    .reindex(evaluation_sample["document_name"], fill_value=0)
    .rename(columns={
        "hazard": "hazards",
        "location": "locations",
        "time": "times",
        "impact": "impacts",
    })
    .reset_index()
)

assert set(annotations["document_id"]) == set(evaluation_sample["document_name"])
assert (annotations["fact_type"] != "no_event").sum() == 108
assert annotations.loc[annotations["fact_type"] == "no_event", "document_id"].nunique() == 2

display(annotation_summary)

fact_type,document_name,hazards,locations,times,impacts
0,PWC 2015 - Updated estimates on cost of Storm ...,2,0,0,2
1,Krausmann 2014 - STREST report on lessons lear...,2,2,1,15
2,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,2,10,1,24
3,Lavenir 2021 - À la recherche d'une jeune fill...,0,0,0,0
4,ABC 2024 - Traffic jams and flight delays due ...,2,3,1,5
5,"EFE 2024 - The DANA storm, live_ The death tol...",2,4,1,8
6,Plag 2014 - Foreword extreme geohazards—a grow...,0,0,0,0
7,Kadir 2014 - The Impact of Natural Disasters o...,2,1,1,6
8,"Cavaleri 2019 - The October 29, 2018 storm in ...",2,5,1,3


## 9. Check the prepared data

These checks confirm the final corpus counts and the structure of the evaluation sample.

In [14]:
# Check the approved corpus and sample decisions before saving
experimental_documents = prepared.loc[
    prepared["experimental_corpus_inclusion"]
]
selected_documents = prepared.loc[prepared["evaluation_selection"]]

assert len(experimental_documents) == 75
assert prepared["evaluation_eligibility"].sum() == 67
assert not prepared["document_name"].duplicated().any()
assert experimental_documents["markdown_available"].all()

assert not prepared.loc[
    prepared["document_name"] == guardian_duplicate,
    "experimental_corpus_inclusion",
].item()
assert prepared.loc[storm_desmond_mask, "experimental_corpus_inclusion"].sum() == 1

assert not selected_documents["length_outlier"].any()
assert len(selected_documents) == 9
assert (selected_documents.groupby("source_type").size() == 3).all()
assert (
    selected_documents.groupby("source_type")["evaluation_length_position"]
    .apply(lambda values: set(values) == {"short", "middle", "long"})
    .all()
)
assert selected_documents["experimental_corpus_inclusion"].all()

expected_selection = {
    "PWC 2015 - Updated estimates on cost of Storm Desmond",
    "Krausmann 2014 - STREST report on lessons learned from recent catastrophic events",
    "AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLUVIAS INTENSAS",
    "Lavenir 2021 - À la recherche d'une jeune fille tombée dans l'Ourthe à Marcourt - L'Avenir",
    "ABC 2024 - Traffic jams and flight delays due to heavy rain and lightning storm in Malaga",
    "EFE 2024 - The DANA storm, live_ The death toll rises to 158",
    "Plag 2014 - Foreword extreme geohazards—a growing threat for a globally interconnected civilization",
    "Kadir 2014 - The Impact of Natural Disasters on Critical Infrastructures - A Domino Effect-based Study",
    "Cavaleri 2019 - The October 29, 2018 storm in Northern Italy – An exceptional event and its modeling",
}
assert set(selected_documents["document_name"]) == expected_selection

print("All preparation checks passed.")
print(f"Experimental corpus: {len(experimental_documents)} documents")
print(f"Eligible for evaluation sampling: {prepared['evaluation_eligibility'].sum()} documents")
print(f"Evaluation sample: {len(selected_documents)} documents")

All preparation checks passed.
Experimental corpus: 75 documents
Eligible for evaluation sampling: 67 documents
Evaluation sample: 9 documents


## 10. Save the prepared document list

I save one CSV with the preparation status of every original PDF. It includes the missing document and excluded duplicate so that the decisions remain visible.

In [15]:
output_columns = [
    "document_name",
    "source_type",
    "word_count",
    "markdown_available",
    "duplicate_status",
    "length_outlier",
    "experimental_corpus_inclusion",
    "evaluation_eligibility",
    "evaluation_selection",
    "evaluation_length_position",
    "exclusion_reason",
]

# Sort by document name so the saved CSV has a stable order
prepared_output = prepared[output_columns].sort_values(
    "document_name",
    na_position="last",
).reset_index(drop=True)

prepared_output.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved {len(prepared_output)} rows to:")
print(OUTPUT_PATH)

Saved 77 rows to:
C:\Users\onssr\Desktop\AIHWS\text2hazard_project\data\prepared_documents.csv


## 11. Preparation summary

The prepared experimental corpus contains 75 documents. The Papatolios PDF is excluded because it has no cleaned Markdown input, and the Guardian-named Storm Desmond copy is excluded because the Treanor version is retained. The original source files remain unchanged.

The eight document-length outliers from Notebook 01 remain in the experimental corpus but are not eligible for the manually annotated quantitative evaluation sample. The final sample contains nine deliberately selected documents:

- **Institutional / technical reports:** PWC (short), Krausmann (middle), and AEMET (long).
- **News / media:** Lavenir (short), ABC (middle), and EFE (long).
- **Scientific papers:** Plag (short), Kadir (middle), and Cavaleri (long).

This sample covers the three reviewed source types and different document lengths, but it is too small and deliberately constructed to represent the complete corpus statistically.

The manual reference annotations for these nine documents are stored in `data/evaluation/manual_annotations.csv` and are used later for evaluation.